In [ ]:
%env RANK=0
%env WORLD_SIZE=1
%env MASTER_ADDR=127.0.0.1
%env MASTER_PORT=2020

In [ ]:
llama_checkpoint_dir = "modified_llama/llama-2-7b"
tokenizer_path = "modified_llama/tokenizer.model"
compression_checkpoint_file = "cv_library/attention_model.pt"
cv_db_dir = ""
max_seq_len = 1024
max_batch_size = 4

In [ ]:
from modified_llama.llama import Llama
import torch

# Create the Llama generator
print("Building generator...")
generator = Llama.build(
    ckpt_dir=llama_checkpoint_dir,
    tokenizer_path=tokenizer_path,
    max_seq_len=max_seq_len,
    max_batch_size=max_batch_size,
)
print("Built generator!")

In [ ]:
from cv_library.compressor import Compressor
from cv_library.loss_functions import sequence_similarity
from cv_hier_storage import ContextVectorHierDB, DBConfig

from pathlib import Path

config = DBConfig([])
cv_db = ContextVectorHierDB(Path(cv_db_dir), config, sequence_similarity)
compressor = Compressor(compression_checkpoint_file)

In [ ]:
query_string = ""

In [ ]:
# Tokenize the content and query, and generate context vectors for them
query_tokens = generator.tokenize(max_seq_len, [("", query_string)])
query_tokens = [l for _, l in query_tokens]

_, query_cvs = generator.generate_context_vectors(query_tokens, len(query_tokens[0]))

In [ ]:
compressed_cvs = compressor.compress(query_cvs[16])
cv_db.search(compressed_cvs)